# M05C: Response Caching

Cache API responses to eliminate duplicate calls and reduce costs.

**Topics:**
- ResponseCache for duplicate elimination
- Cache key design: prompt + instructions + model

---

## 🔧 Step 1: Setup

In [ ]:
import os
import hashlib
from pathlib import Path
from dotenv import load_dotenv
import openai

load_dotenv(dotenv_path=Path("..") / ".env")

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"

print(f"✅ Setup complete: Using {MODEL}!")

---

## 🎯 The Problem

When processing many prompts, duplicates waste money — you're paying for the same API call twice.

Caching stores the response the first time, then returns it instantly for duplicates.

---

## 💾 Response Caching

Same input = same output. Cache the response the first time, return it instantly for duplicates.

**Key idea:** Cache by **prompt + instructions + model** — any difference produces a different answer.

In [ ]:
class ResponseCache:
    """Cache API responses to avoid duplicates."""
    
    def __init__(self):
        self.cache = {}
        self.hits = 0
        self.misses = 0
    
    def _make_key(self, prompt, instructions, model):
        """Create cache key from prompt + instructions + model."""
        text = f"{model}|{prompt.strip()}|{instructions.strip()}"
        return hashlib.md5(text.encode()).hexdigest()
    
    def get(self, prompt, instructions, model=MODEL):
        """Get cached response if exists."""
        key = self._make_key(prompt, instructions, model)
        if key in self.cache:
            self.hits += 1
            return self.cache[key]
        self.misses += 1
        return None
    
    def set(self, prompt, instructions, response, model=MODEL):
        """Store response in cache."""
        key = self._make_key(prompt, instructions, model)
        self.cache[key] = response
    
    def stats(self):
        """Get cache statistics."""
        total = self.hits + self.misses
        hit_rate = (self.hits / total * 100) if total else 0
        return {
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": hit_rate,
            "size": len(self.cache)
        }


# --------------------------------------------------------------
print("✅ ResponseCache ready")

### Demo: Caching in Action

In [ ]:
print("💾 CACHING DEMO")
print("="*60)

cache = ResponseCache()
prompts = ["What is AI?", "What is ML?", "What is AI?"]
instructions = "5 words. Be concise."

for prompt in prompts:
    cached = cache.get(prompt, instructions)
    if cached is not None:
        print(f"[CACHE HIT ] {prompt} → {cached}")
    else:
        response = client.responses.create(
            model=MODEL,
            input=prompt,
            instructions=instructions
        )
        result = response.output_text.strip()
        cache.set(prompt, instructions, result)
        print(f"[API CALL  ] {prompt} → {result}")

print("\n" + "="*60)
print(f"Cache Stats: {cache.stats()}")
print("="*60)

---

### 💪 Your Turn: Email Classifier with Caching

Use the `ResponseCache` class to classify a batch of support emails. Some are duplicates — your cache should skip those.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise: Email Classifier with Caching
# --------------------------------------------------------------
# Objective: Classify emails using caching to skip duplicate API calls.

# Hint: The demo cell above shows the cache pattern — adapt it for email classification.

emails = [
    "URGENT: Account security alert!",
    "Your order has shipped",
    "URGENT: Account security alert!",  # duplicate
    "Win a FREE iPhone now!",
    "Your order has shipped",  # duplicate
    "Meeting reminder: Tomorrow at 2pm",
    "URGENT: Suspicious login detected",
    "Thanks for your purchase!",
    "Win a FREE iPhone now!",  # duplicate
    "Your subscription will renew soon"
]

instructions = "Classify as: urgent, transactional, spam, or routine. One word."

# TODO 1: Create a ResponseCache
# TODO 2: Loop through emails:
#    - Check cache before calling the API
#    - If cache miss, classify with the API and store the result
# TODO 3: Print each email and its classification
# TODO 4: Print cache stats — how many API calls did you save?

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**💾 Caching Eliminates Duplicate API Calls:**
- Cache key = prompt + instructions + model
- Check cache before every API call, store after
- Track hit rate to measure savings

---

### 📍 Next Step

**M05D: Capstone #2 — Production Support Bot** — Combine budget controls, context management, and response caching into a production support chatbot.

---

## 🔧 Troubleshooting

**Cache not hitting?**
- Check if prompts are EXACTLY identical
- Instructions must match too
- Leading/trailing whitespace is stripped; internal differences still count

**Still having issues?**
- Copy any error message and paste it into ChatGPT, Claude, Gemini, or Grok — they're great at debugging
- Re-watch the lecture for this module
- Post to the Q&A with your error message and output

---